In [2]:
import cx_Oracle
import pandas as pd
from sklearn.preprocessing import StandardScaler
from datetime import datetime
from utils import helpers
import os

In [ ]:
df=helpers.rebalancing_by_period('2021-01-01','2025-01-01',5)

In [ ]:
df

In [ ]:
import os
print(os.getcwd()) 

In [ ]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings('ignore')
pd.options.display.float_format='{:.4%}'.format

In [ ]:
start='2016-01-01'
end='2019-12-30'

assets=['AAPL']
assets.sort()

data=yf.download(assets,start=start,end=end)
data=data.loc[:,('Adj Close',slice(None))]
data.columns=assets

In [ ]:
data=[
    ("S_VAL_PB_NEW","PB(市净率)","每股股价与每股净资产的比率"),
    ("S_VAL_PE_TTM","PE(市盈率)","每股股价与每股净资产的比率(静态), 算法：市盈率（PE，TTM）=总市值/净利润（TTM），其中：总市值=股价*总股本；总股本=A股股价*公司发行在外普通股总数（多地上市股份总和）")
]

In [ ]:
df=pd.DataFrame(data,columns=["英文名称","中文名称","因子含义"])

In [ ]:
df

In [ ]:
df.to_csv("utils/factor.csv",index=False,encoding="utf_8_sig")

In [ ]:
df1=pd.read_csv("utils/factor.csv")

In [ ]:
df1

In [ ]:
for index,row in df1.iterrows():
    englist_name=row['英文名称']
    combined_info=f"因子名称:{row['中文名称']},因子含义:{row['因子含义']}"
    
    print(englist_name)
    print(combined_info)

In [ ]:
project_root = os.getcwd()  #获取当前目录路径
path_name="prem_value"

In [ ]:
res_path=os.path.join(project_root,"result",path_name)

In [ ]:
res_path

In [ ]:
from pathlib import Path
py_file=Path(res_path)/f"{englist_name}.py"
py_file

In [ ]:
import cx_Oracle
import pandas as pd

JYLH_DB = {
    'username': 'jylh',
    'password': 'jylh',
    'host': '10.6.60.114:1521',
    'service': 'wind'
}

WIND_DB = {
    'username': 'wind',
    'password': 'wind',
    'host': '10.6.60.114:1521',
    'service': 'wind'
}

def execute_query(connection_params, sql, params=None):
    """
    执行SQL查询并返回DataFrame。
    
    该函数封装了数据库连接的创建、查询执行、数据获取和连接关闭的全过程，
    确保资源被正确释放，并使用参数化查询防止SQL注入<websource>source_group_web_1</websource>。

    参数:
        connection_params (dict): 包含数据库连接信息的字典。
        sql (str): SQL查询语句。
        params (tuple, optional): 用于参数化查询的参数元组，默认为None。

    返回:
        pd.DataFrame: 查询结果组成的DataFrame。
    """
    # 构建连接字符串
    dsn = f"{connection_params['username']}/{connection_params['password']}@{connection_params['host']}/{connection_params['service']}"
    connection = cx_Oracle.connect(dsn)
    cursor = None
    try:
        cursor = connection.cursor()
        # 使用参数化查询，将SQL结构与数据分离，有效防止SQL注入
        if params:
            cursor.execute(sql, params)
        else:
            cursor.execute(sql)
        # 获取所有结果
        columns = [col[0] for col in cursor.description]
        results = cursor.fetchall()
        
        return pd.DataFrame(results,columns=columns)
    finally:
        # 确保游标和连接最终被关闭，防止资源泄漏
        if cursor:
            cursor.close()
        connection.close()


In [ ]:
#申万行业变更表
sql = "select STOCKCODE,SWLEVEL1CODE,BEGINDATE,ENDDATE from JY_QY_STOCK_INDUSTRY"
df = execute_query(JYLH_DB, sql)

In [ ]:
df

In [ ]:
#兼容代码表
sql = "select NAME,CODE from JY_QY_DIMENSION"
df1 = execute_query(JYLH_DB, sql)

In [ ]:
df1

In [ ]:
df2=pd.merge(df,df1,left_on=['SWLEVEL1CODE'],right_on=['CODE'],how='left')

In [ ]:
df2=df2.drop_duplicates()
df3=df2.drop(columns=['SWLEVEL1CODE','CODE']).reset_index(drop=True)

In [ ]:
df3

In [ ]:
# 假设您的 DataFrame 名为 df
# 1. 按照 股票代码、名称、开始日期 进行排序，把相同的组聚在一起
df_sorted = df3.sort_values(by=['STOCKCODE', 'NAME', 'BEGINDATE']).reset_index(drop=True)

# 2. 判断当前行的时间是否与上一行连续或重叠
# 为了处理复杂的重叠，先计算同一股票、同一名称下的累积最大结束时间
df_sorted['cummax_end'] = df_sorted.groupby(['STOCKCODE', 'NAME'])['ENDDATE'].cummax()
df_sorted['prev_max_end'] = df_sorted.groupby(['STOCKCODE', 'NAME'])['cummax_end'].shift()

# 3. 如果当前行的开始时间 > 之前记录的最大结束时间，说明时间断开了，属于新的区间
is_new_interval = (df_sorted['BEGINDATE'] > df_sorted['prev_max_end']) | df_sorted['prev_max_end'].isna()

# 4. 生成区间分组ID并聚合取 min 和 max
df_sorted['interval_id'] = is_new_interval.cumsum()
df_merged = df_sorted.groupby(['interval_id', 'STOCKCODE', 'NAME']).agg({
    'BEGINDATE': 'min',
    'ENDDATE': 'max'
}).reset_index()

# 5. 整理最终的列顺序，并按照原始习惯排序
df_merged = df_merged[['STOCKCODE', 'BEGINDATE', 'ENDDATE', 'NAME']]
df_merged = df_merged.sort_values(by=['STOCKCODE', 'BEGINDATE']).reset_index(drop=True)
df_merged=df_merged.drop_duplicates(subset=['STOCKCODE', 'BEGINDATE', 'ENDDATE'],keep='first').reset_index(drop=True)


In [ ]:
df_merged.loc[df_merged['ENDDATE']==99999999,'ENDDATE']=20301230

In [ ]:
df_merged

In [ ]:
df_stock = pd.read_csv(
        '/home/quant/zc/finance_deal/qlib_data/price_data0821/instruments/stock_code.txt',
        sep='\s+',
        header=None,
        names=['col1', 'col2', 'col3']
    )

In [ ]:
col_mapping={item.split('.')[0]:item for item in list(df_stock['col1'])}

In [ ]:
df_merged['STOCKCODE']=df_merged['STOCKCODE'].map(col_mapping)

In [ ]:
df_merged=df_merged.dropna()

In [ ]:
df_merged.to_csv("inst_stock.csv",encoding='utf-8-sig', index=False)

In [ ]:
df_merged.head(10)

### 基金持仓数据分析

In [ ]:
#申万行业变更表
sql = "select F_INFO_WINDCODE,F_INFO_FULLNAME,F_INFO_NAME,F_INFO_FIRSTINVESTTYPE,F_INFO_SETUPDATE,F_INFO_MATURITYDATE,F_INFO_FIRSTINVESTSTYLE from ChinaMutualFundDescription"
# sql="select S_INFO_WINDCODE,F_PRT_ENDDATE,S_INFO_STOCKWINDCODE,ANN_DATE,REPORT_TYPE from ChinaMutualFundStockPortfolio"
df = execute_query(WIND_DB, sql)

In [ ]:
# 查看有多少只基金，以及有多少个描述字段
print("数据形状:", df.shape)

# 查看前5行数据，直观感受内容
print("\n前5行数据:")
print(df.head())

In [ ]:
# 打印所有列名
print("\n所有字段名:")
print(df.columns.tolist())

# 查看每个字段的数据类型和非空值数量
print("\n字段信息:")
print(df.info())

In [ ]:
# 查看基金类型的分布情况
print("\n基金类型分布:")
print(df['F_INFO_FIRSTINVESTTYPE'].value_counts())
# print(df['REPORT_TYPE'].value_counts())

print("\n基金风格分布:")
print(df['F_INFO_FIRSTINVESTSTYLE'].value_counts())




In [3]:
test1=pd.read_csv("fund/processed_fund_holdings.csv")

In [4]:
test1.head()

,S_INFO_WINDCODE,F_PRT_ENDDATE,S_INFO_STOCKWINDCODE,ANN_DATE,REPORT_TYPE,F_INFO_WINDCODE,F_INFO_FULLNAME,F_INFO_NAME,F_INFO_FIRSTINVESTTYPE,F_INFO_SETUPDATE,F_INFO_MATURITYDATE,F_INFO_FIRSTINVESTSTYLE,EFFECTIVE_DATE
0,000652.OF,2005-06-30,002049.SZ,2005-08-24,中/年报,000652.OF,博时裕隆灵活配置混合型证券投资基金,博时裕隆A,混合型,20140603,NaN,灵活配置型,2005-08-24
1,000652.OF,2005-06-30,002047.SZ,2005-08-24,中/年报,000652.OF,博时裕隆灵活配置混合型证券投资基金,博时裕隆A,混合型,20140603,NaN,灵活配置型,2005-08-24
2,000652.OF,2005-06-30,002046.SZ,2005-08-24,中/年报,000652.OF,博时裕隆灵活配置混合型证券投资基金,博时裕隆A,混合型,20140603,NaN,灵活配置型,2005-08-24
3,000652.OF,2005-06-30,002045.SZ,2005-08-24,中/年报,000652.OF,博时裕隆灵活配置混合型证券投资基金,博时裕隆A,混合型,20140603,NaN,灵活配置型,2005-08-24
4,000652.OF,2005-06-30,002043.SZ,2005-08-24,中/年报,000652.OF,博时裕隆灵活配置混合型证券投资基金,博时裕隆A,混合型,20140603,NaN,灵活配置型,2005-08-24


In [5]:
df_fund=test1[['S_INFO_WINDCODE','F_INFO_FULLNAME','F_INFO_NAME','F_INFO_FIRSTINVESTTYPE','F_INFO_FIRSTINVESTSTYLE']]
df_fund=df_fund.drop_duplicates()

In [6]:
df_fund

,S_INFO_WINDCODE,F_INFO_FULLNAME,F_INFO_NAME,F_INFO_FIRSTINVESTTYPE,F_INFO_FIRSTINVESTSTYLE
0,000652.OF,博时裕隆灵活配置混合型证券投资基金,博时裕隆A,混合型,灵活配置型
65,003715.OF,宝盈消费主题灵活配置混合型证券投资基金,宝盈消费主题,混合型,灵活配置型
104,004666.OF,长城久嘉创新成长灵活配置混合型证券投资基金,长城久嘉创新成长A,混合型,灵活配置型
138,050001.OF,博时价值增长证券投资基金,博时价值增长,混合型,平衡型
157,050004.OF,博时精选混合型证券投资基金,博时精选A,混合型,混合型
...,...,...,...,...,...
3606557,023908.OF,安信上证科创板综合指数增强型发起式证券投资基金,安信上证科创综指增强A,股票型,增强指数型
3808968,023913.OF,富国上证科创板综合价格指数增强型证券投资基金,富国上证科创板综合价格指数增强A,股票型,增强指数型
3810435,024377.OF,易方达科智量化选股股票型发起式证券投资基金,易方达科智量化选股A,股票型,股票型
3810627,024450.OF,易方达成长进取混合型证券投资基金,易方达成长进取A,混合型,混合型


In [7]:
import pandas as pd
import re
import jieba
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import normalize

# 1. 文本清洗：去除冗余的基金后缀、通用词汇（让模型专注于核心“主题词”）
def clean_fund_name(name):
    name = str(name)
    # 去除 A/C/E 等份额字母后缀
    name = re.sub(r'[A-Z]+$', '', name)
    # 去除通用的类型词（因为这些特征已经由 F_INFO_FIRSTINVESTTYPE 提供）
    name = re.sub(r'(ETF|LOF|FOF|混合|灵活配置|配置|股票|债券|增强|指数|发起式|证券|投资|基金|回报|精选|优选)', '', name)
    return name

df_fund['CLEAN_NAME'] = df_fund['F_INFO_NAME'].apply(clean_fund_name)

# 2. 中文分词：将名称切分成有意义的词语
def tokenize_text(text):
    # 使用 jieba 进行中文分词，以空格连接
    return " ".join(jieba.lcut(text))

df_fund['TOKEN_NAME'] = df_fund['CLEAN_NAME'].apply(tokenize_text)

# 3. 提取文本特征：使用词级别的 TF-IDF
# max_df=0.9 忽略在90%基金中都出现的词，min_df=2 忽略只出现过1次的生僻词
vectorizer = TfidfVectorizer(max_df=0.9, min_df=2, max_features=300)
text_sparse = vectorizer.fit_transform(df_fund['TOKEN_NAME'])

# 4. 提取类别特征：One-Hot 编码
cat_features = pd.get_dummies(df_fund[['F_INFO_FIRSTINVESTSTYLE']])
cat_sparse = csr_matrix(cat_features.values)

# 5. 特征合并与归一化（核心优化点）
# 将类别特征的权重适当放大（例如乘以1.5），防止被维度较多的文本特征淹没
X_combined = hstack([cat_sparse * 1, text_sparse])
# 对所有样本进行 L2 归一化，这对高维稀疏特征的 K-Means 聚类效果提升极大
X_normalized = normalize(X_combined)

# 6. 重新执行 K-Means 聚类
kmeans = KMeans(n_clusters=40, random_state=42, n_init='auto')
df_fund['Cluster_Opt'] = kmeans.fit_predict(X_normalized)

# 7. 查看优化后的分布情况
print("【优化后】各个类别的基金数量分布：\n", df_fund['Cluster_Opt'].value_counts())


Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.689 seconds.
Prefix dict has been built successfully.


【优化后】各个类别的基金数量分布：
 Cluster_Opt
1     954
2     790
0     134
16     78
39     61
12     56
37     41
19     37
26     37
28     36
18     36
15     35
13     33
10     27
25     24
23     24
31     23
11     23
14     22
22     19
27     19
21     19
17     19
36     18
38     18
3      17
7      16
32     16
24     12
33     12
35     11
8      10
29     10
9      10
5       9
30      9
6       7
20      6
34      6
4       2
Name: count, dtype: int64


In [8]:
df_fund[df_fund['Cluster_Opt'] == 34]

,S_INFO_WINDCODE,F_INFO_FULLNAME,F_INFO_NAME,F_INFO_FIRSTINVESTTYPE,F_INFO_FIRSTINVESTSTYLE,CLEAN_NAME,TOKEN_NAME,Cluster_Opt
773049,004128.OF,新疆前海联合泳隆灵活配置混合型证券投资基金,前海联合泳隆A,混合型,灵活配置型,前海联合泳隆,前 海 联合 泳隆,34
847052,002780.OF,新疆前海联合泓鑫灵活配置混合型证券投资基金,前海联合泓鑫A,混合型,灵活配置型,前海联合泓鑫,前 海 联合 泓 鑫,34
850449,004693.OF,新疆前海联合泳隽灵活配置混合型证券投资基金,前海联合泳隽A,混合型,灵活配置型,前海联合泳隽,前 海 联合 泳隽,34
966268,005671.OF,新疆前海联合研究优选灵活配置混合型证券投资基金,前海联合研究优选A,混合型,灵活配置型,前海联合研究,前 海 联合 研究,34
1193838,004809.OF,新疆前海联合润丰灵活配置混合型证券投资基金,前海联合润丰A,混合型,灵活配置型,前海联合润丰,前 海 联合 润丰,34
1196971,005933.OF,新疆前海联合先进制造灵活配置混合型证券投资基金,前海联合先进制造A,混合型,灵活配置型,前海联合先进制造,前 海 联合 先进 制造,34


In [9]:
df_fund.to_csv("fund/fund_cluster.csv",encoding='utf-8-sig',index=False)